In [8]:
import cv2
import os
import torch
import clip
from PIL import Image

In [9]:
def crop_image(crop_percent,img_path,output_folder):
    img_name = img_path.split("/")[-1]
    
    if not (img_name.endswith(".jpg") or img_name.endswith(".png")):
        print(f"{img_name} not cropped \n only .png or .jpg supported")
        return 
    
    img = cv2.imread(img_path)
    h,w = img.shape[:2]
    
    # compute crop coordinates
    top = int(h * crop_percent)
    bottom = int(h * (1 - crop_percent))
    left = int(w * crop_percent)
    right = int(w * (1 - crop_percent))
    
    
    
    cropped_img = img[top:bottom, left:right]
    print(f"{img_name}: cropped size = {cropped_img.shape}")

    name, ext = os.path.splitext(img_name)
    save_path = os.path.join(output_folder, f"{name}_{crop_percent}_pct_crop{ext}")
    success = cv2.imwrite(save_path, cropped_img)
    if not success:
        print(f"Failed to save {save_path}")

    print(f"{img_name} successfully cropped \n")
    return cropped_img
    

In [10]:
def blur_image(kernel_size,sigmaX,img_path,output_folder):
    img_name = img_path.split("/")[-1]
    
    if not (img_name.endswith(".jpg") or img_name.endswith(".png")):
        print(f"{img_name} not cropped \n only .png or .jpg supported")
        
        return 
    
    img = cv2.imread(img_path)
    blurred_img = cv2.GaussianBlur(img,kernel_size,sigmaX)
    
    os.makedirs(output_folder, exist_ok=True)
    name, ext = os.path.splitext(img_name)
    save_path = os.path.join(output_folder, f"{name}_blurred{ext}")
    success = cv2.imwrite(save_path, blurred_img)
    if not success:
        print(f"Failed to save {save_path}")

    print(f"{img_name} successfully blurred \n")
    
    return blurred_img
    
    

In [11]:
input_folder = "images/original/"
output_folder = "images/cropped/"
os.makedirs(output_folder, exist_ok=True)

In [12]:
crop_percent = 0.1
for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    crop_image(crop_percent, img_path, output_folder)

dog_02.png: cropped size = (342, 512, 3)
dog_02.png successfully cropped 

dog_01.png: cropped size = (342, 512, 3)
dog_01.png successfully cropped 

dog_03.png: cropped size = (384, 512, 3)
dog_03.png successfully cropped 

dog_05.png: cropped size = (258, 512, 3)
dog_05.png successfully cropped 

__pycache__ not cropped 
 only .png or .jpg supported
caption.py not cropped 
 only .png or .jpg supported
dog_04.png: cropped size = (512, 384, 3)
dog_04.png successfully cropped 



In [13]:
kernel_size = (5,5)
sigmaX = 0
blurred_output_folder= "images/blurred"
for img_name in os.listdir(input_folder):
    img_path = os.path.join(input_folder, img_name)
    blur_image(kernel_size,sigmaX,img_path,blurred_output_folder)

dog_02.png successfully blurred 

dog_01.png successfully blurred 

dog_03.png successfully blurred 

dog_05.png successfully blurred 

__pycache__ not cropped 
 only .png or .jpg supported
caption.py not cropped 
 only .png or .jpg supported
dog_04.png successfully blurred 



### similarity measure
*   Loops over images + captions
*   Computes similarity for original vs cropped
*   Computes drop in similarity (misalignment metric)

In [ ]:
sys.path.append("images/original")  
from caption import captions  

original_folder = "images/original/"
cropped_folder = "images/cropped/"
blurred_folder = "images/blurred/"

# load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def compute_similarity(image_path, text):
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    text_tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text_tokens)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        similarity = (image_features @ text_features.T).item()
    return similarity

# collect results
results = {}

for img_name, caption in captions.items():
    name, ext = os.path.splitext(img_name)
    
    # original image
    orig_path = os.path.join(original_folder, img_name)
    orig_sim = compute_similarity(orig_path, caption)
    
    # cropped image
    crop_name = f"{name}_{crop_percent}_pct_crop{ext}"
    crop_path = os.path.join(cropped_folder, crop_name)
    crop_sim = compute_similarity(crop_path, caption)
    
    # blurred image
    blur_name = f"{name}_blurred{ext}"
    blur_path = os.path.join(blurred_folder, blur_name)
    blur_sim = compute_similarity(blur_path, caption)
    
    # store results
    results[img_name] = {
        "original": orig_sim,
        "cropped": crop_sim,
        "blurred": blur_sim,
        "drop_crop": orig_sim - crop_sim,
        "drop_blur": orig_sim - blur_sim
    }

# print results
for img_name, res in results.items():
    print(f"{img_name}: Original={res['original']:.4f}, Cropped={res['cropped']:.4f}, "
          f"Blurred={res['blurred']:.4f}, Drop_crop={res['drop_crop']:.4f}, Drop_blur={res['drop_blur']:.4f}")


dog_04.png: Original=0.2953, Cropped=0.2882, Blurred=0.2958, Drop_crop=0.0071, Drop_blur=-0.0005
dog_03.png: Original=0.2877, Cropped=0.3023, Blurred=0.2866, Drop_crop=-0.0146, Drop_blur=0.0011
dog_02.png: Original=0.2476, Cropped=0.2456, Blurred=0.2559, Drop_crop=0.0020, Drop_blur=-0.0083
dog_05.png: Original=0.3360, Cropped=0.3235, Blurred=0.3291, Drop_crop=0.0125, Drop_blur=0.0069
dog_01.png: Original=0.2665, Cropped=0.2633, Blurred=0.2640, Drop_crop=0.0032, Drop_blur=0.0025
